# core

> Connect to jupygate-hosted kernels and turn execution into concise text

In [ ]:
#| default_exp core

clikernel is the LLM side of a two-process design: jupygate runs all the time and hosts the kernels; clikernel starts and stops with each conversation and holds nothing but a pointer. This module is the whole client: gateway resolution from `gateways.toml`, the concise-text rendering contract (ANSI-stripped, capped tracebacks), delivery of `startup.py` and `inspectors.py` into freshly created kernels, and `Client` — connect (create or attach), execute, interrupt, restart, stop, list. The MCP and CLI frontends are thin faces over `Client`; nothing here creates or kills a kernel except when asked, and the one scoped kill is opt-in: a kernel created with `auto=True` belongs to its client, ended by that client's next `connect` or by the frontend on the way out.


In [ ]:
#| export
import os, tomllib
from fastcore.utils import *
from fastcore.nbio import render_text
from fastcore.xdg import xdg_config_home
from jupyasyncclient import JupyAsyncMultiKernelManager, DeadKernelError

In [ ]:
from fastcore.test import *
import asyncio, socket, tempfile, time
from jupygate.core import create_app, serve


## Configuration

`$XDG_CONFIG_HOME/clikernel/` holds three files, all optional: `startup.py` (run in every kernel clikernel creates), `inspectors.py` (installed right after), and `gateways.toml`, which names remote gateways so that tokens never travel as tool arguments (tool args persist in transcripts). A `host` is resolved in one of three ways: empty means the default local gateway (`$CLIKERNEL_HOST` or `http://127.0.0.1:8787`), a URL is used as given, and anything else is looked up as a `gateways.toml` name:

    [gateways.solveit]
    url = "https://solveit.example.com/gate"
    token_env = "SOLVEIT_TOKEN"


In [ ]:
#| export
DEFAULT_URL = 'http://127.0.0.1:8787'
MAXLEN = 180 # Most characters shown per displayed line

def cfg_dir():
    "The clikernel config directory"
    return xdg_config_home()/'clikernel'

def gateways(cfgdir=None):
    "Named gateways from `gateways.toml`: `{name: {url, token | token_env}}`"
    p = (Path(cfgdir) if cfgdir else cfg_dir())/'gateways.toml'
    return tomllib.loads(p.read_text()).get('gateways', {}) if p.exists() else {}

def resolve(host='', cfgdir=None):
    "`(url, token)` for `host`: empty = the default local gateway, a URL = itself, else a `gateways.toml` name"
    if not host: return os.environ.get('CLIKERNEL_HOST', DEFAULT_URL), os.environ.get('CLIKERNEL_TOKEN')
    if '://' in host: return host, os.environ.get('CLIKERNEL_TOKEN')
    g = gateways(cfgdir).get(host)
    if g is None: raise ValueError(f"unknown gateway {host!r}: not a URL, and not in {cfg_dir()/'gateways.toml'}")
    return g['url'], g.get('token') or os.environ.get(g.get('token_env','')) or None


In [ ]:
cfgd = Path(tempfile.mkdtemp())
(cfgd/'gateways.toml').write_text('[gateways.solveit]\nurl = "https://s.example.com/gate"\ntoken = "T"\n')
test_eq(resolve(), (DEFAULT_URL, os.environ.get('CLIKERNEL_TOKEN')))
test_eq(resolve('http://h:1/p'), ('http://h:1/p', os.environ.get('CLIKERNEL_TOKEN')))
test_eq(resolve('solveit', cfgd), ('https://s.example.com/gate', 'T'))
test_fail(lambda: resolve('nope', cfgd), contains='unknown gateway')

Kernels emit tracebacks IPython-formatted but ANSI-colored, and a cell magic's transformed source can echo its whole payload on one over-long line. fastcore's `render_text` owns the cleanup mechanism (ANSI stripping always on; `tb_maxlen` caps traceback lines while keeping `File `/`Cell ` locations and the final exception-message chunk whole); clikernel's `execute` just picks the display budget it passes as `tb_maxlen`.


## Startup and inspectors


Both config files are delivered *as source* to kernels clikernel creates — the kernel may be on another machine, so no file path can be assumed there, and the local config stays authoritative either way. `startup.py` runs first, wrapped so `__file__` is bound to its local path during the run and gone afterwards (matching v1, which used `%run -i`); its output is the text `connect` returns. `inspectors.py` installs second, with v1's contract intact: the file may define `inspect` and/or a list `inspectors`; each is called once per cell before it runs — 1-arg inspectors get the cell's AST, 2-arg ones get `(tree, src)` with the raw source for lexical checks. An inspector may return a note (printed before the cell's output), raise `RuleBlock` (provided in the file's namespace; the cell does not run), or return None. Any other exception is an inspector bug: noted, and the cell runs — fail-open, because a crashed inspector must never masquerade as a policy block. A file that fails to *load* raises, failing the create call: refusing to start beats running uninspected.


In [ ]:
#| export
def _startup_src(src, path):
    "The startup file's source wrapped so `__file__` is bound to its path during the run, and absent after"
    return f'''__file__ = {str(path)!r}
try: exec(compile({src!r}, __file__, 'exec'))
finally: del __file__'''

In [ ]:
#| export
_INSP_RUNNER = r'''
import inspect as _clik_inspect
import sys as _clik_sys
from IPython.core.error import InputRejected
class RuleBlock(InputRejected):
    "Raise from an inspector to deliberately block a cell; any other inspector exception is a bug, and fails open"

class _ClikInspect:
    "Calls each inspector once per cell: 1-arg get the AST, 2-arg also the raw source"
    def __init__(self, fs): self.fs = fs
    def visit(self, tree):
        fr, n = _clik_sys._getframe(), 0
        while fr:
            n += fr.f_code.co_name == 'run_cell_async'
            fr = fr.f_back
        if n > 1: return tree  # nested run_cell: cell replayed by a tool (%nbrun etc.), not typed
        for f in self.fs:
            try:
                note = f(tree, _clik_src) if len(_clik_inspect.signature(f).parameters) > 1 else f(tree)
                if note: print(note, end='')
            except InputRejected: raise
            except Exception as e: print(f'inspector error (cell runs anyway): {e!r}')
        return tree

def _clik_stash(info):
    global _clik_src
    _clik_src = info.raw_cell

def _clik_install(src):
    ns = dict(RuleBlock=RuleBlock)
    exec(compile(src, 'inspectors.py', 'exec'), ns)
    fs = list(ns.get('inspectors') or [])
    if callable(ns.get('inspect')): fs.append(ns['inspect'])
    if fs:
        ip = get_ipython()
        ip.events.register('pre_run_cell', _clik_stash)
        ip.ast_transformers.append(_ClikInspect(fs))
_clik_src = ''
'''

def _inspector_setup(src):
    "Kernel-side source installing the inspectors defined in `src`; a load failure raises, failing the create call"
    return _INSP_RUNNER + f'\n_clik_install({src!r})'

## The client

`Client` is one conversation's whole state: a gateway manager and the current kernel's ws client — nothing else. `connect` resolves the host, verifies the gateway is reachable (a bad URL or token fails at the connect call, not three calls later), then either attaches to an existing kernel by id prefix (as found: nothing is run) or creates a fresh one, runs `startup.py`, and installs `inspectors.py`. Every verb answers in plain text, because the model is the caller. Nothing here ever stops a kernel implicitly — dropping a connection, switching kernels, and clikernel exiting all leave kernels running — with one deliberate exception: a kernel created with `auto=True` is scoped to its client, and that client's next `connect` ends it. `stop` is the only other kill, and it needs an explicit call.

In [ ]:
#| export
STATE_LOST = 'NOTE: the kernel restarted with a fresh interpreter: all session state is lost (imports, variables, monkeypatches). Redo any setup the task still needs.'

class Client:
    "One conversation's connection: a gateway manager, and the current kernel"
    def __init__(self, cfgdir=None): self.cfgdir,self.mgr,self.kc,self.kid,self.auto = cfgdir,None,None,None,False

    async def _use(self, mgr, kid):
        "Point at `kid` on `mgr`, dropping any previous ws (never the kernel)"
        if self.kc: await self.kc.aclose()
        if self.mgr is not None and self.mgr is not mgr: await self.mgr.aclose()
        self.mgr,self.kid,self.kc = mgr,kid,mgr.client(kid)
        self.kc.start_channels()
        await self.kc.wait_for_ready(timeout=30)

In [ ]:
#| export
@patch
async def connect(self:Client, host='', kernel='', auto=False):
    "Connect to a gateway (create a kernel, or attach to `kernel` by id prefix); the pointer aims at it afterwards; `auto` marks a created kernel as client-scoped"
    note = ''
    if self.auto:   # scoped to this client: any new connect ends it, even one already dead
        try: await self.stop()
        except Exception as e:
            self.kc,self.kid,self.auto = None,None,False
            note = f'\nnote: stopping the auto kernel failed ({e})'
    url,tok = resolve(host, self.cfgdir)
    mgr = JupyAsyncMultiKernelManager(url, token=tok)
    ks = await mgr.list_kernels()   # verify reachability and auth now, loudly
    if kernel:
        kid = first(k['id'] for k in ks if k['id'].startswith(kernel))
        if not kid: raise ValueError(f'no kernel matching {kernel!r} on {url}: {[k["id"][:8] for k in ks]}')
        await self._use(mgr, kid)
        return f'connected to existing kernel {kid} on {url}' + note
    kw = dict(cwd=os.getcwd(), env=dict(os.environ)) if not host else {}   # the default gateway is local by definition: kernels start where, and as, the conversation lives (cwd and environment - so kernel-side tools that key state to the conversation, like llmdojo's doc-state, resolve it correctly)
    kid = await mgr.start_kernel(**kw)
    await self._use(mgr, kid)
    out = ''
    d = Path(self.cfgdir) if self.cfgdir else cfg_dir()
    if (p := d/'startup.py').exists(): out = await self.execute(_startup_src(p.read_text(), p))
    if (p := d/'inspectors.py').exists():
        res = await self.execute(_inspector_setup(p.read_text()))
        if res:   # a load failure is fatal: refusing to start beats running uninspected
            await self.mgr.shutdown_kernel(kid)
            self.kc,self.kid = None,None
            raise RuntimeError(f'inspectors.py failed to load; kernel stopped:\n{res}')
    self.auto = auto
    return f'created kernel {kid} on {url}' + (f'\n{out}' if out.strip() else '') + note

@patch
async def execute_outs(self:Client, code):
    "Run `code` in the current kernel; nbformat-style output dicts (or a protocol note string)"
    if not self.kc: return 'no kernel: call `connect` first'
    try: return await self.kc.run(code)
    except DeadKernelError: return 'NOTE: the kernel process died. `connect` to create or attach to another.'

@patch
async def execute(self:Client, code):
    "Run `code` in the current kernel; concise rendered text of its outputs"
    r = await self.execute_outs(code)
    return r if isinstance(r, str) else render_text(r, tb_maxlen=MAXLEN)

In [ ]:
#| export
@patch
async def list_kernels(self:Client, host=''):
    "One line per kernel on the gateway (the current one if `host` is empty and connected)"
    if host or self.mgr is None:
        url,tok = resolve(host, self.cfgdir)
        mgr = JupyAsyncMultiKernelManager(url, token=tok)
        ks = await mgr.list_kernels()
        await mgr.aclose()
    else: ks = await self.mgr.list_kernels()
    if not ks: return 'no kernels'
    def _l(k): return f"{k['id']}  {k.get('execution_state','?')}  connections={k.get('connections','?')}" + ('  <- current' if k['id']==self.kid else '')
    return '\n'.join(_l(k) for k in ks)

@patch
async def stop(self:Client, kernel=''):
    "Stop a kernel by id prefix (the current one if empty): the only way anything is ever killed"
    if not self.mgr: return 'no gateway: call `connect` first'
    if not kernel and not self.kid: return 'no current kernel: pass an id from `list_kernels`'
    kid = self.kid if not kernel else first(k['id'] for k in await self.mgr.list_kernels() if k['id'].startswith(kernel))
    if not kid: return f'no kernel matching {kernel!r}'
    await self.mgr.shutdown_kernel(kid)
    if kid == self.kid:
        await self.kc.aclose()
        self.kc,self.kid,self.auto = None,None,False
    return f'stopped kernel {kid}'

@patch
async def restart(self:Client):
    "Restart the current kernel: same id, fresh interpreter"
    if not self.kc: return 'no kernel: call `connect` first'
    await self.mgr.restart_kernel(self.kid)
    await self.kc.wait_for_ready(timeout=30)
    return STATE_LOST

@patch
async def interrupt(self:Client):
    "SIGINT the current kernel's running cell"
    if not self.kc: return 'no kernel: call `connect` first'
    await self.mgr.interrupt_kernel(self.kid)
    return 'interrupt sent'

@patch
async def aclose(self:Client):
    "Drop connections; kernels are left exactly as they are"
    if self.kc: await self.kc.aclose()
    if self.mgr: await self.mgr.aclose()
    self.mgr,self.kc,self.kid = None,None,None

All of it live, against an in-thread jupygate with an isolated config dir: `startup.py` prints a banner (and proves `__file__` is bound, v1-style), and `inspectors.py` has a 2-arg noting rule, a blocking rule, and a deliberately buggy one to show fail-open:

In [ ]:
def free_port():
    with socket.socket() as s:
        s.bind(('127.0.0.1', 0))
        return s.getsockname()[1]

gport = free_port()
gserver = serve(create_app(), port=gport, in_thread=True)
os.environ['CLIKERNEL_HOST'] = f'http://127.0.0.1:{gport}'

cfgd = Path(tempfile.mkdtemp())
(cfgd/'startup.py').write_text('import sys\nbase = 42\nprint("ready, file", __file__.rsplit("/",1)[-1])')
insp_src = '''
import ast
def note_sleep(tree, src):
    if 'time.sleep' in src: return 'note: sleeping\\n'
def block_sub(tree):
    if any(isinstance(n, ast.Import) and any(a.name=='subprocess' for a in n.names) for n in ast.walk(tree)):
        raise RuleBlock('no subprocess in kernels')
def buggy(tree, src):
    if 'trigger_bug' in src: raise TypeError('oops')
inspectors = [note_sleep, block_sub, buggy]
'''
(cfgd/'inspectors.py').write_text(insp_src)

c = Client(cfgd)
res = await c.connect()
res

'created kernel ba63d7ddc35147cebd5d798f16c940f4 on http://127.0.0.1:62104\nready, file startup.py\n'

The three inspector behaviors, and the ordinary execute contract — state persists, multiple outputs get tagged sections, `input()` fails fast in-band (`allow_stdin=False`), and tracebacks come back clean:

In [ ]:
test_eq(await c.execute('base'), '42')
test_eq(await c.execute('import os; os.getcwd()'), repr(os.getcwd()))  # kernel born in the conversation's cwd
noted = await c.execute('import time; time.sleep(0.01); 7')
assert noted.startswith('<stdout>\nnote: sleeping') and '7' in noted
buggy = await c.execute('trigger_bug = 1; 8')
assert 'inspector error (cell runs anyway)' in buggy and '8' in buggy   # fail-open: noted, and the cell ran
blocked = await c.execute('import subprocess')
assert 'no subprocess in kernels' in blocked
test_eq(await c.execute('base'), '42')  # blocked cell really did not run, state intact
res = await c.execute('input("never")')
assert 'StdinNotImplementedError' in res and '\x1b' not in res
res


'---------------------------------------------------------------------------\nStdinNotImplementedError                  Traceback (most recent call last)\nCell In[9], line 1\n----> 1 input("never")\n\nFile ~/aai-ws/ipymini/ipymini/term/io.py:148, in _thread_local_input(prompt)\n    146 if handler is None or not allow:\n    147     msg = "raw_input was called, but this frontend does not support input requests."\n--> 148     raise StdinNotImplementedError(msg)\n    149 return handler(str(prompt), False)\n\nStdinNotImplementedError: raw_input was called, but this frontend does not support input requests.'

The persistence story, which is the point of the whole design: drop the client (conversation over), and the kernel is still there — a fresh client (next conversation) lists it, attaches by id prefix, and finds its state. Attach runs neither startup nor inspectors: the kernel is taken as found.

In [ ]:
kid1 = c.kid
await c.aclose()                      # conversation ends; nothing is stopped
c2 = Client(cfgd)
print(await c2.list_kernels())        # a fresh client sees it
res = await c2.connect(kernel=kid1[:8])
test_eq(await c2.execute('base'), '42')   # state survived the "conversation boundary"
res

ba63d7ddc35147cebd5d798f16c940f4  alive  connections=0


'connected to existing kernel ba63d7ddc35147cebd5d798f16c940f4 on http://127.0.0.1:62104'

`restart` keeps the id but resets the interpreter; `interrupt` stops a too-long cell without losing state; `stop` is the only explicit kill, and `list_kernels` confirms it:

In [ ]:
res = await c2.restart()
assert 'state is lost' in res
test_eq(await c2.execute('"base" in dir()'), 'False')   # fresh interpreter, same kernel id
test_eq(c2.kid, kid1)

task = asyncio.create_task(c2.execute('import time; time.sleep(60); "never"'))
await asyncio.sleep(0.3)
print(await c2.interrupt())
res = await task
assert 'KeyboardInterrupt' in res
test_eq(await c2.execute('21*2'), '42')                 # state survived the interrupt

print(await c2.stop())
test_eq(await c2.stop(), 'no current kernel: pass an id from `list_kernels`')
await c2.list_kernels()

interrupt sent
stopped kernel ba63d7ddc35147cebd5d798f16c940f4


'no kernels'

An *auto* kernel is one a frontend created on demand rather than by an explicit decision — the MCP `execute` tool does this when it's called with no kernel connected. `connect(auto=True)` records it, and the scope is the client: connecting anywhere else stops the auto kernel immediately, and the MCP frontend stops it at process exit. Explicitly created or attached kernels keep the persistence contract above: only `stop` ends them.

In [ ]:
await c2.connect(auto=True)
assert c2.auto
akid = c2.kid
res = await c2.connect()                  # connecting elsewhere stops the auto kernel at once
assert not c2.auto
assert akid[:8] not in await c2.list_kernels()
await c2.stop()                           # the explicitly created kernel still needs its explicit stop

A dead auto kernel must never block connecting onward: the reap tolerates a kernel that is already gone, and the connect reply says what happened rather than failing or staying silent.

In [ ]:
await c2.connect(auto=True)
await c2.mgr.shutdown_kernel(c2.kid)   # dies out from under us (gateway restart, idle reaping, ...)
res = await c2.connect()
assert 'note:' in res                  # connect succeeded and reported the failed reap
assert c2.kid and not c2.auto
await c2.stop()

When a kernel dies behind the client's back (gateway restart, an out-of-band stop), `execute` answers with the concise died-note rather than a traceback: jupyasyncclient signals `DeadKernelError` whichever way it discovers the death — the liveness probe during execution, or the websocket reconnect finding the kernel 404.

In [ ]:
await c2.connect()
await c2.mgr.shutdown_kernel(c2.kid)   # killed behind our back
res = await c2.execute('1+1')
assert res.startswith('NOTE')          # a concise note, not a traceback
await c2.connect()
test_eq(await c2.execute('1+1'), '2')  # and connecting again recovers
await c2.stop()

In [ ]:
#|hide
await c2.aclose()
gserver.should_exit = True
del os.environ['CLIKERNEL_HOST']

In [ ]:
#|hide
#|eval: false
import nbdev; nbdev.nbdev_export()